# Deterministic UMAP Parameter Exploration

Compare four historical Hyrax UMAPs with fixed-seed controls and parameter variants. New embeddings are stored separately under `umap_explorations`; existing `runN/udbN_E` artifacts are never changed.

The reported trustworthiness and neighbor-retention values assess how faithfully the 2D projection preserves neighborhoods from the inference vectors. They do **not** establish whether the learned representation is scientifically or semantically good.

## 1. Environment and imports

Set the profile before importing `research_paths`. On Delta this defaults to `delta`; set `HYRAX_PROFILE=local` before starting the kernel to use the local fixture instead.

In [ ]:
import os
from pathlib import Path

PROFILE = os.environ.get("HYRAX_PROFILE", "delta")
os.environ["HYRAX_PROFILE"] = PROFILE

from IPython.display import display

from research_paths import paths
from umap_parameter_explorer import (
    ExperimentRef,
    UmapParams,
    artifacts_table,
    diagnose_runs,
    ensure_controls,
    evaluate_visualizer_selection,
    historical_runs,
    manifest_table,
    open_visualizer,
    plot_grid,
    plot_selection_results,
    preflight,
    run_all,
    run_selected,
    runs_table,
    selection_results_table,
    session_directory,
)

print(f"Profile: {paths.profile}")
print(f"Run root: {paths.hyrax_runs}")
print(f"Exploration root: {paths.umap_explorations}")

## 2. Choose experiments and a session

Replace these four `(run, expt)` pairs with the experiments you want to inspect. Keep the same session name to reuse identical completed variants; change it when you want a separate exploration.

In [ ]:
if PROFILE == "local":
    EXPERIMENTS = [ExperimentRef(run=1, expt=expt) for expt in range(1, 5)]
else:
    EXPERIMENTS = [
        ExperimentRef(run=5, expt=7),
        ExperimentRef(run=8, expt=4),
        ExperimentRef(run=8, expt=5),
        ExperimentRef(run=8, expt=6),
    ]

SESSION = os.environ.get("UMAP_EXPLORATION_SESSION", "umap_parameter_sweep_01")
RANDOM_SEED = 42
EVALUATION_SIZE = 2_000
DIAGNOSTIC_K = 15
OUTPUT_ROOT = Path(os.environ.get("UMAP_EXPLORATION_ROOT", paths.umap_explorations))
SESSION_DIR = session_directory(OUTPUT_ROOT, SESSION)

print(f"Session directory: {SESSION_DIR}")

## 3. Preflight

Resolve every UDB config/log, inference directory, and historical 2D UMAP before writing anything. Parameters absent from an older config resolve to UMAP defaults (`min_dist=0.1`, `metric='euclidean'`).

In [ ]:
artifacts = preflight(EXPERIMENTS, paths.hyrax_runs)
display(artifacts_table(artifacts))

## 4. Historical overview

The historical UMAPs are referenced in place. Their old fit samples and random states were not recorded, so they are labeled `seed unknown`. Diagnostics use one fixed evaluation sample per experiment.

In [ ]:
historical = historical_runs(artifacts)
plot_grid(historical, suptitle="Historical UMAPs", ncols=2)

historical = diagnose_runs(
    historical,
    output_root=OUTPUT_ROOT,
    session=SESSION,
    seed=RANDOM_SEED,
    evaluation_size=EVALUATION_SIZE,
    diagnostic_k=DIAGNOSTIC_K,
)
display(runs_table(historical))

## 5. Deterministic controls

Each control uses the experiment's historical parameter values, a fixed UMAP/transform seed, and a fixed fit sample whose size matches the source UDB config. The first execution computes results; later identical calls reuse them.

In [ ]:
controls = ensure_controls(
    artifacts,
    output_root=OUTPUT_ROOT,
    session=SESSION,
    seed=RANDOM_SEED,
    evaluation_size=EVALUATION_SIZE,
    diagnostic_k=DIAGNOSTIC_K,
)
plot_grid(controls, suptitle="Deterministic controls", ncols=2)
display(runs_table(controls))

In [ ]:
historical_vs_control = [
    run
    for ref in EXPERIMENTS
    for run in (historical[ref], controls[ref])
]
plot_grid(
    historical_vs_control,
    suptitle="Historical (left) vs deterministic control (right)",
    ncols=2,
)

## 6. Edit parameters and rerun

Change only the cell below. `n_components` remains fixed at 2. Set `RUN_ALL_FOUR=False` while iterating cheaply on one focused experiment; set it to `True` to apply the same parameters to all four.

Useful starting metrics for latent vectors are `euclidean`, `cosine`, `correlation`, and `manhattan`. UMAP validates the name during fitting.

In [ ]:
PARAMS = UmapParams(
    n_neighbors=30,
    min_dist=0.05,
    metric="cosine",
)

FOCUS_EXPERIMENT = EXPERIMENTS[0]
RUN_ALL_FOUR = False

In [ ]:
if RUN_ALL_FOUR:
    variants = run_all(
        artifacts,
        PARAMS,
        output_root=OUTPUT_ROOT,
        session=SESSION,
        seed=RANDOM_SEED,
        label="parameter variant",
        evaluation_size=EVALUATION_SIZE,
        diagnostic_k=DIAGNOSTIC_K,
    )
else:
    variants = {
        FOCUS_EXPERIMENT: run_selected(
            artifacts[FOCUS_EXPERIMENT],
            PARAMS,
            output_root=OUTPUT_ROOT,
            session=SESSION,
            seed=RANDOM_SEED,
            label="parameter variant",
            evaluation_size=EVALUATION_SIZE,
            diagnostic_k=DIAGNOSTIC_K,
        )
    }

plot_grid(variants, suptitle="Current parameter variant", ncols=2)
display(runs_table(variants))

## 7. Focused interactive visualization

Open any historical, control, or variant result through the existing Hyrax `visualize` verb. Image thumbnails default off to keep the comparison responsive.

In [ ]:
FOCUS_RUN = variants.get(FOCUS_EXPERIMENT, controls[FOCUS_EXPERIMENT])
pane, viz = open_visualizer(
    FOCUS_RUN,
    display_images=False,
    width=550,
    height=550,
)

## 8. Manual selection completeness and purity

Use the lasso tool to draw an ellipse-like region on the interactive UMAP above. Each lasso is one operating point: **completeness** is the fraction of all known targets captured by the selection, while **purity** is the fraction of selected objects with known labels that are targets. Objects with missing or unmatched labels are excluded from both denominators and reported through the selected-label coverage columns.

For an unbiased validation check, draw the region without showing the target labels. If the target overlay influenced the selection, interpret the result as descriptive and potentially optimistic.

In [ ]:
import static_umap_metrics as selection_metrics

selection_metrics.init_tabular_stack()
selection_context = selection_metrics.load_path_context(profile=PROFILE)
selection_catalog = selection_metrics.load_sample_catalog(selection_context, name="all")
selection_overlay_groups = selection_metrics.build_default_overlay_groups(selection_catalog)

# Edit this group or replace TARGET_RULES with your own overlay dictionaries.
TARGET_GROUP = "recent_merger_flags"
if TARGET_GROUP not in selection_overlay_groups:
    raise KeyError(
        f"Target group {TARGET_GROUP!r} is unavailable. "
        f"Available groups: {sorted(selection_overlay_groups)}"
    )
TARGET_RULES = selection_overlay_groups[TARGET_GROUP]

display(selection_metrics.summarize_overlays(selection_catalog, TARGET_RULES))

The list below is deliberately in memory only. To compare another UMAP, assign that result to `FOCUS_RUN`, rerun the interactive visualization cell, draw a new lasso, give it a unique `SELECTION_NAME`, and rerun the scoring cell.

In [ ]:
selection_evaluations = []

After drawing a lasso, name and score it below. Re-running this cell appends another evaluation. The selected-object table is long-form: each selected object appears once for every target rule.

In [ ]:
if len(viz.points_id) == 0:
    raise ValueError(
        "No UMAP points are selected. Use the lasso-select tool, draw a region, "
        "and confirm that the table beside the UMAP updates before scoring."
    )
print(f"Selected UMAP points: {len(viz.points_id):,}")

SELECTION_NAME = (
    f"run{FOCUS_RUN.artifact.ref.run}_expt{FOCUS_RUN.artifact.ref.expt}"
    " candidate region 1"
)
selection_evaluation = evaluate_visualizer_selection(
    FOCUS_RUN,
    viz,
    selection_catalog,
    TARGET_RULES,
    selection_name=SELECTION_NAME,
)
selection_evaluations.append(selection_evaluation)

display(selection_evaluation.summary)
display(selection_evaluation.selected_objects)

In [ ]:
selection_comparison = selection_results_table(selection_evaluations)
display(selection_comparison)
selection_figure, selection_axes = plot_selection_results(selection_evaluations)
selection_figure

## 9. Manifest and cache reuse

The manifest is rebuilt from completed per-result `provenance.json` files. Repeating the same source, sample, parameters, seed, and package-version combination returns the existing directory rather than creating another copy.

In [ ]:
display(manifest_table(SESSION_DIR))

reopened = run_selected(
    artifacts[FOCUS_EXPERIMENT],
    PARAMS,
    output_root=OUTPUT_ROOT,
    session=SESSION,
    seed=RANDOM_SEED,
    label="parameter variant",
    evaluation_size=EVALUATION_SIZE,
    diagnostic_k=DIAGNOSTIC_K,
)
print(f"Reused cached result: {reopened.reused}")
print(reopened.result_dir)

## Interpretation reminders

- Vary one parameter at a time first. Lower `n_neighbors` emphasizes local structure; higher values emphasize broader structure.
- Lower `min_dist` permits tighter clumps; higher values spread nearby points out.
- Changing `metric` changes what counts as near in the inference space, so compare both the plot and the diagnostics.
- A manual lasso contributes one completeness-purity point; a collection of hand-drawn selections is a comparison of operating points, not a threshold-generated curve.
- Stable, high-fidelity 2D structure across reasonable settings is evidence that the display is not merely a fragile UMAP accident. It still does not prove the training objective learned the scientific distinctions you care about; that requires labels, overlays, or a downstream validation task.